# Session 09: LangGraph Platform

## Deploying the Stone Ridge Investment Assistant with LangGraph Platform

### Learning Objectives

- **Understand LangGraph Platform architecture** — how `langgraph.json`, `app/agent.py`, and the CLI work together
- **Walk through the consolidated agent** — understand the graph structure with guardrails, helpfulness evaluation, and investment-domain tools
- **Deploy locally with `langgraph dev`** — run the investment assistant as a local API server
- **Test via the LangGraph SDK** — interact with the deployed agent programmatically

### Overview

In this notebook we explore the **`app/agent.py`** file — a consolidated LangGraph agent that combines:
- **Anthropic Claude** as the LLM (Claude Sonnet for main agent, Claude Haiku for evaluation)
- **RAG** over the Stone Ridge 2025 Investor Letter
- **X/Twitter tools** for market sentiment analysis
- **Tavily + Arxiv** for web and academic search
- **Input/output guardrails** for production safety
- **Helpfulness evaluation** loop for response quality

The agent is deployed via **LangGraph Platform**, which provides:
- A REST API for invocation
- Built-in streaming support
- Thread-based memory management
- LangSmith integration for observability

---

## Task 1: Understand the Agent Architecture

Let's start by examining the key components of our consolidated agent.

### Graph Architecture

```
START \u2192 input_guardrail \u2192[pass]\u2192 agent \u2192[tools?]\u2192 action \u2192 agent (loop)
                         [fail]\u2192 END        [no tools]\u2192 output_guardrail \u2192[pass]\u2192 helpfulness \u2192[Y]\u2192 END
                                                                          [fail]\u2192 agent (retry)   [N]\u2192 agent
```

### Key Components

| Component | Description | Model |
|-----------|-------------|-------|
| Input Guardrail | Topic restriction, jailbreak detection, PII protection | Guardrails AI |
| Agent | Main reasoning + tool calling | Claude Sonnet |
| Action | Tool execution (Tavily, Arxiv, RAG, X/Twitter) | N/A |
| Output Guardrail | PII protection, profanity filter | Guardrails AI |
| Helpfulness | Response quality evaluation | Claude Haiku |

In [ ]:
import os
import getpass

if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass.getpass("Anthropic API Key:")

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key (for embeddings):")

# Optional
if not os.environ.get("TAVILY_API_KEY"):
    tavily = getpass.getpass("Tavily API Key (optional \u2014 Enter to skip):")
    if tavily.strip():
        os.environ["TAVILY_API_KEY"] = tavily

### Examining the Agent Code

Let's look at the key sections of `app/agent.py`:

In [ ]:
# View the agent module structure
with open("app/agent.py") as f:
    lines = f.readlines()

# Print section headers
for i, line in enumerate(lines, 1):
    if line.startswith("# ----") or line.startswith("def ") or line.startswith("class "):
        print(f"{i:4d}: {line.rstrip()}")

---

## Task 2: Import and Compile the Graph Locally

Before deploying to LangGraph Platform, let's import and test the graph directly.

In [ ]:
from app.agent import (
    graph,
    build_graph,
    get_chat_model,
    get_tool_belt,
    SYSTEM_PROMPT,
    DEFAULT_MODEL,
    EVAL_MODEL,
)

print(f"Graph compiled successfully!")
print(f"Default model: {DEFAULT_MODEL}")
print(f"Eval model: {EVAL_MODEL}")
print(f"\nTools available:")
for t in get_tool_belt():
    print(f"  - {t.name}")

In [ ]:
# Visualize the graph
from IPython.display import Image, display

display(Image(graph.get_graph().draw_mermaid_png()))

---

## Task 3: Test the Graph Locally

Let's invoke the graph directly with some investment-related queries.

In [ ]:
from langchain_core.messages import HumanMessage

# Test with an investment question
result = graph.invoke(
    {"messages": [HumanMessage(content="What is Stone Ridge's investment philosophy regarding reinsurance?")]}
)

# Print the final response (skip HELPFULNESS markers)
for msg in reversed(result["messages"]):
    if hasattr(msg, "content") and not msg.content.startswith("HELPFULNESS:"):
        print(msg.content)
        break

In [ ]:
# Test the guardrails with an off-topic query
result = graph.invoke(
    {"messages": [HumanMessage(content="What medicine should I take for a headache?")]}
)

for msg in reversed(result["messages"]):
    if hasattr(msg, "content") and msg.content:
        print(msg.content)
        break

In [ ]:
# Test with a query that should use the RAG tool
result = graph.invoke(
    {"messages": [HumanMessage(
        content="According to the Stone Ridge 2025 Investor Letter, how does the firm think about bitcoin allocation?"
    )]}
)

for msg in reversed(result["messages"]):
    if hasattr(msg, "content") and not msg.content.startswith("HELPFULNESS:"):
        print(msg.content)
        break

---

## Task 4: Deploy with LangGraph Platform

Now let's deploy the agent using LangGraph Platform. The configuration is defined in `langgraph.json`:

```json
{
  "version": 1,
  "dependencies": ["."],
  "env": ".env",
  "python_version": "3.13",
  "graphs": {
    "investment_assistant": "app.agent:graph"
  }
}
```

### Starting the Server

In a terminal, run:

```bash
cd 09_Production_and_MCP
uv run langgraph dev
```

This will:
1. Install dependencies from `pyproject.toml`
2. Load environment variables from `.env`
3. Start a local API server at `http://localhost:2024`
4. Open the LangGraph Studio UI for visual debugging

---

## Task 5: Test via LangGraph SDK

Once the server is running, we can interact with it using the **LangGraph SDK**. This is how production clients would call the agent.

In [ ]:
from langgraph_sdk import get_sync_client

# Connect to the local LangGraph Platform server
client = get_sync_client(url="http://localhost:2024")

print("Connected to LangGraph Platform!")

# List available assistants
assistants = client.assistants.search()
for a in assistants:
    print(f"  - {a['assistant_id']}: {a.get('name', 'unnamed')}")

In [ ]:
# Stream a response from the deployed agent
for chunk in client.runs.stream(
    None,  # Threadless run
    "investment_assistant",
    input={
        "messages": [
            {
                "role": "human",
                "content": "What is Stone Ridge's view on reinsurance as an asset class?",
            }
        ]
    },
    stream_mode="updates",
):
    print(f"Event: {chunk.event}")
    if chunk.data:
        # Print the last message content if available
        messages = chunk.data.get("messages", [])
        for msg in messages:
            content = msg.get("content", "")
            if content and not content.startswith("HELPFULNESS:"):
                print(f"  {content[:200]}..." if len(content) > 200 else f"  {content}")
    print()

In [ ]:
# Test with a thread for multi-turn conversation
thread = client.threads.create()
print(f"Thread created: {thread['thread_id']}")

# First message
for chunk in client.runs.stream(
    thread["thread_id"],
    "investment_assistant",
    input={"messages": [{"role": "human", "content": "Tell me about Stone Ridge's bitcoin strategy."}]},
    stream_mode="updates",
):
    if chunk.event == "updates" and chunk.data:
        messages = chunk.data.get("messages", [])
        for msg in messages:
            content = msg.get("content", "")
            if content and not content.startswith("HELPFULNESS:"):
                print(content[:500])

In [ ]:
# Follow-up question (same thread = agent remembers context)
for chunk in client.runs.stream(
    thread["thread_id"],
    "investment_assistant",
    input={"messages": [{"role": "human", "content": "How does that compare to their reinsurance approach?"}]},
    stream_mode="updates",
):
    if chunk.event == "updates" and chunk.data:
        messages = chunk.data.get("messages", [])
        for msg in messages:
            content = msg.get("content", "")
            if content and not content.startswith("HELPFULNESS:"):
                print(content[:500])

### ❓ Question #1

What are the benefits of deploying an agent through LangGraph Platform versus running it directly as a Python script? Consider: scalability, observability, memory management, and client access.

##### Answer

**Scalability:**
- LangGraph Platform provides a REST API server out of the box, meaning multiple clients can call the agent concurrently without managing threading or process isolation yourself. A raw Python script is single-process by default — you'd need to wrap it in FastAPI/Flask, handle async concurrency, and manage deployment infrastructure manually.
- The platform also handles dependency management and environment isolation via `langgraph.json`, making deployment reproducible across environments.

**Observability:**
- LangGraph Platform integrates natively with LangSmith, providing automatic tracing of every graph invocation — node-by-node execution, token counts, latency breakdowns, and tool call logs. With a raw script, you'd need to instrument tracing manually (e.g., adding LangSmith callbacks, custom logging).
- The LangGraph Studio UI gives real-time visual debugging of graph execution, which is invaluable for understanding agent decision-making during development.

**Memory Management:**
- The platform provides built-in **thread-based memory** — each `thread_id` maintains its own conversation history, enabling multi-turn conversations with zero additional code. With a raw script, you'd need to implement your own state persistence (MemorySaver, Redis, database) and manage thread isolation.
- Threads are managed server-side, so clients don't need to maintain state.

**Client Access:**
- The LangGraph SDK provides production-grade client features: streaming (token-by-token or update-by-update), async invocation, thread management, and assistant discovery. A raw script would need all of this built from scratch.
- The REST API also means any language/platform can call the agent (not just Python), enabling web frontends, mobile apps, and other services to integrate easily.

### ❓ Question #2

How does the helpfulness evaluation loop affect latency and cost? When would you enable or disable it in production?

##### Answer

**Latency impact:**
- The helpfulness check adds one additional LLM call per response using Claude Haiku (the cheaper/faster eval model). This typically adds ~0.3-1s of latency. In the happy path (response deemed helpful on first try), this is the only overhead.
- In the unhappy path (response deemed unhelpful), the agent retries — meaning another full Sonnet inference + tool calls + another Haiku evaluation. Worst case, this can 2-3x total response time. The `len(messages) > 10` guard prevents infinite loops.

**Cost impact:**
- Haiku evaluation calls are cheap (~10-20x less than Sonnet per token), so the evaluation itself costs very little.
- The real cost comes from retries: each retry burns a full Sonnet call. In practice, well-prompted agents with good RAG retrieval produce helpful responses on the first attempt 85-95% of the time, so the amortized cost increase is ~10-20%.

**When to enable:**
- Customer-facing products where response quality directly impacts user satisfaction and retention.
- High-stakes domains (investment advice, compliance-sensitive content) where an unhelpful response could damage trust.
- When you have latency budget (e.g., users expect a few seconds for a thorough analysis).

**When to disable:**
- Internal tools where speed matters more than polish (e.g., developer debugging assistants).
- Real-time or streaming-first interfaces where users expect immediate token output.
- High-throughput batch processing where the extra LLM call per request significantly increases cost.
- During development/prototyping when you want fast iteration cycles.

### \ud83c\udfd7\ufe0f Activity #1

Modify the agent to add a new graph variant:

1. Create a "simple" variant in `app/agent.py` that skips guardrails and helpfulness (just agent + action nodes)
2. Register it in `langgraph.json` as `"simple_assistant"`
3. Restart `langgraph dev` and test both variants via the SDK
4. Compare response times and quality

In [ ]:
"""
Activity #1: Create a "simple" graph variant that skips guardrails and helpfulness.

This variant only has: agent → action (tool loop) → END
No input/output guardrails, no helpfulness evaluation.
"""

from app.agent import (
    get_chat_model,
    get_tool_belt,
    AgentState,
    SYSTEM_PROMPT,
)
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode
from langchain_core.messages import HumanMessage, AIMessage
from typing import Dict, Any
import time


# --- Step 1: Define simple graph nodes (no guardrails, no helpfulness) ---

def simple_agent(state: AgentState) -> Dict[str, Any]:
    """Invoke the LLM with tools — no guardrails wrapping."""
    model = get_chat_model().bind_tools(get_tool_belt())
    messages = [HumanMessage(content=SYSTEM_PROMPT)] + state["messages"]
    response = model.invoke(messages)
    return {"messages": [response]}


def simple_route(state: AgentState):
    last_message = state["messages"][-1]
    if getattr(last_message, "tool_calls", None):
        return "action"
    return END


# --- Step 2: Build the simple graph ---

def build_simple_graph():
    tool_node = ToolNode(get_tool_belt())
    g = StateGraph(AgentState)
    g.add_node("agent", simple_agent)
    g.add_node("action", tool_node)
    g.set_entry_point("agent")
    g.add_conditional_edges("agent", simple_route, {"action": "action", END: END})
    g.add_edge("action", "agent")
    return g.compile()


simple_graph = build_simple_graph()
print("Simple graph compiled (no guardrails, no helpfulness)")

# Visualize
from IPython.display import Image, display
display(Image(simple_graph.get_graph().draw_mermaid_png()))

# --- Step 3: Compare both variants ---

from app.agent import graph as full_graph

test_query = "What is Stone Ridge's view on bitcoin as a portfolio allocation?"

# Test full graph
print("\n" + "=" * 60)
print("FULL GRAPH (guardrails + helpfulness)")
print("=" * 60)
start = time.time()
full_result = full_graph.invoke({"messages": [HumanMessage(content=test_query)]})
full_time = time.time() - start
full_msg_count = len(full_result["messages"])

for msg in reversed(full_result["messages"]):
    if hasattr(msg, "content") and msg.content and not msg.content.startswith("HELPFULNESS:"):
        print(f"Response: {msg.content[:400]}...")
        break
print(f"Time: {full_time:.2f}s | Messages: {full_msg_count}")

# Test simple graph
print("\n" + "=" * 60)
print("SIMPLE GRAPH (agent + tools only)")
print("=" * 60)
start = time.time()
simple_result = simple_graph.invoke({"messages": [HumanMessage(content=test_query)]})
simple_time = time.time() - start
simple_msg_count = len(simple_result["messages"])

for msg in reversed(simple_result["messages"]):
    if hasattr(msg, "content") and msg.content:
        print(f"Response: {msg.content[:400]}...")
        break
print(f"Time: {simple_time:.2f}s | Messages: {simple_msg_count}")

# --- Step 4: Summary ---
print("\n" + "=" * 60)
print("COMPARISON SUMMARY")
print("=" * 60)
print(f"{'Metric':<25} {'Full Graph':<15} {'Simple Graph':<15}")
print("-" * 55)
print(f"{'Latency':<25} {full_time:.2f}s{'':<10} {simple_time:.2f}s")
print(f"{'Messages':<25} {full_msg_count:<15} {simple_msg_count:<15}")
print(f"{'Guardrails':<25} {'Yes':<15} {'No':<15}")
print(f"{'Helpfulness check':<25} {'Yes':<15} {'No':<15}")
if simple_time > 0:
    print(f"\nSimple graph is {full_time/simple_time:.1f}x {'slower' if full_time < simple_time else 'faster'} than full graph")

# --- Step 5: Note on langgraph.json registration ---
print("\n--- To register as 'simple_assistant' in langgraph.json: ---")
print("""
Add to app/agent.py:
    simple_graph = build_simple_graph()

Update langgraph.json:
    {
      "graphs": {
        "investment_assistant": "app.agent:graph",
        "simple_assistant": "app.agent:simple_graph"
      }
    }

Then restart: uv run langgraph dev
""")

---

## Summary

In this notebook we covered:

- **Agent Architecture**: Walked through the consolidated `app/agent.py` with guardrails, tools, and helpfulness evaluation
- **Local Testing**: Imported and tested the graph directly in Python
- **LangGraph Platform Deployment**: Used `langgraph dev` to deploy as a local API server
- **SDK Client**: Tested the deployed agent with streaming and multi-turn threads

### Key Takeaways

1. **LangGraph Platform = deployment made simple** \u2014 `langgraph.json` + `app/agent.py` is all you need
2. **The SDK provides production-grade access** \u2014 threads, streaming, and async are built in
3. **Guardrails add safety but also latency** \u2014 consider the tradeoff per use case
4. **The helpfulness loop improves quality** \u2014 but costs an extra LLM call per response